In [29]:
import importlib
from scripts import data_loader

importlib.reload(data_loader)

print(data_loader.BASE_PATH)
print(data_loader.BASE_PATH.exists())
print((data_loader.BASE_PATH / "separations").exists())

/Users/destin/Projects/federal-doge-study/data
True
True


In [34]:
from pathlib import Path
import pandas as pd
from scripts.data_loader import load_federal_data

base = Path("../data/separations")

dfs = []
for file in base.glob("*.txt"):
    df = pd.read_csv(file, sep="|")
    
    parts = file.stem.split("_")
    year = int(parts[1][:4])
    month = int(parts[1][4:])
    
    df["year"] = year
    df["month"] = month
    
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

/var/folders/pz/csmxncdx0_db1nd5f3qk4y340000gn/T/ipykernel_63159/3601915448.py:9: DtypeWarning: Columns (8,30,51) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep="|")
/var/folders/pz/csmxncdx0_db1nd5f3qk4y340000gn/T/ipykernel_63159/3601915448.py:9: DtypeWarning: Columns (8,51) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep="|")


In [35]:
sep_time = df.groupby(["year", "month"]).size()
sep_time

year  month
2024  1        19115
2025  1        22290
2026  1        39630
dtype: int64

In [36]:
sep_agency = df.groupby("agency_subelement").size().sort_values(ascending=False)
sep_agency.head(10)

agency_subelement
VETERANS HEALTH ADMINISTRATION             9856
INTERNAL REVENUE SERVICE                   6117
TRANSPORTATION SECURITY ADMINISTRATION     1891
MILITARY TREATMENT FACILITIES UNDER DHA    1798
SOCIAL SECURITY ADMINISTRATION             1538
U.S. ARMY CORPS OF ENGINEERS               1521
ENVIRONMENTAL PROTECTION AGENCY            1405
FOREST SERVICE                             1350
DEFENSE HUMAN RESOURCES ACTIVITY           1277
NATIONAL PARK SERVICE                      1266
dtype: int64

In [37]:
share_by_age = df["age_bracket"].value_counts(normalize=True)
share_by_age


age_bracket
65 OR MORE      0.185722
60-64           0.185673
55-59           0.130030
50-54           0.086950
35-39           0.081237
40-44           0.080114
30-34           0.075609
45-49           0.067600
25-29           0.065194
20-24           0.039094
LESS THAN 20    0.002777
Name: proportion, dtype: float64

In [38]:
share_by_tenure = df["tenure_code"].value_counts(normalize=True)
share_by_tenure

tenure_code
1    0.505023
1    0.149425
2    0.109881
0    0.064475
2    0.048763
3    0.045517
0    0.040149
3    0.036656
*    0.000111
Name: proportion, dtype: float64

# Look at separations post 2025

In [40]:
df["post"] = (df["year"] >= 2025).astype(int)
df.groupby("year").size()

year
2024    19115
2025    22290
2026    39630
dtype: int64

In [41]:
change = df.groupby(["agency_subelement", "post"]).size().unstack().fillna(0)
change["diff"] = change[1] - change[0]
change.sort_values("diff", ascending=False).head(10)

post,0,1,diff
agency_subelement,,,
VETERANS HEALTH ADMINISTRATION,2509.0,7347.0,4838.0
INTERNAL REVENUE SERVICE,865.0,5252.0,4387.0
ENVIRONMENTAL PROTECTION AGENCY,100.0,1305.0,1205.0
DEFENSE HUMAN RESOURCES ACTIVITY,45.0,1232.0,1187.0
U.S. ARMY CORPS OF ENGINEERS,291.0,1230.0,939.0
TRANSPORTATION SECURITY ADMINISTRATION,516.0,1375.0,859.0
IMMIGRATION AND CUSTOMS ENFORCEMENT,119.0,873.0,754.0
SMALL BUSINESS ADMINISTRATION,124.0,855.0,731.0
FEDERAL EMERGENCY MANAGEMENT AGENCY,269.0,986.0,717.0


In [42]:
df.groupby(["age_bracket", "post"]).size().unstack().fillna(0)

post,0,1
age_bracket,,
20-24,1003,2165
25-29,1565,3718
30-34,1798,4329
35-39,1871,4712
40-44,1839,4653
45-49,1435,4043
50-54,1568,5478
55-59,2121,8416
60-64,2862,12184


In [44]:
agg = df.groupby(["agency_subelement", "post"]).size().reset_index(name="separations")

In [45]:
import statsmodels.api as sm
import pandas as pd

agg = df.groupby(["agency_subelement", "post"]).size().reset_index(name="separations")

# simple model
agg["intercept"] = 1
model = sm.OLS(agg["separations"], agg[["intercept", "post"]]).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            separations   R-squared:                       0.015
Model:                            OLS   Adj. R-squared:                  0.013
Method:                 Least Squares   F-statistic:                     12.68
Date:                Wed, 25 Mar 2026   Prob (F-statistic):           0.000389
Time:                        10:55:17   Log-Likelihood:                -6264.1
No. Observations:                 859   AIC:                         1.253e+04
Df Residuals:                     857   BIC:                         1.254e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
intercept     47.9073     17.815      2.689      0.0